# KhazinaSmart — Exploratory Data Analysis

Full visual analysis of the Walmart sales dataset using Plotly interactive charts.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/walmart_clean.csv', parse_dates=['Date'])
print(f'Loaded: {df.shape}')

Loaded: (29000, 16)


## Chart 1 — Weekly Sales Distribution

In [2]:
fig = px.histogram(df, x='Weekly_Sales', nbins=100,
                   title='Weekly Sales Distribution across all Stores & Departments',
                   color_discrete_sequence=['#1a1a2e'])
mean_s = df['Weekly_Sales'].mean()
median_s = df['Weekly_Sales'].median()
fig.add_vline(x=mean_s, line_dash='dash', line_color='#e94560', annotation_text=f'Mean: {mean_s:,.0f}')
fig.add_vline(x=median_s, line_dash='dot', line_color='#f39c12', annotation_text=f'Median: {median_s:,.0f}')
fig.update_layout(height=450, template='plotly_white')
fig.show()

**Key insights:** Sales are right-skewed with a long tail — most store/dept combos sell under 20K/week, but high-volume departments pull the mean above the median.

## Chart 2 — Sales Trend Over Time

In [3]:
weekly = df.groupby(['Date', 'IsHoliday'])['Weekly_Sales'].sum().reset_index()
fig2 = px.line(weekly, x='Date', y='Weekly_Sales', color='IsHoliday',
               color_discrete_map={True: '#e94560', False: '#4a9eff'},
               title='Total Weekly Sales Trend (2010–2012)')
fig2.update_layout(height=450, template='plotly_white')
fig2.show()

**Key insights:** Clear year-end seasonality peaks in November-December. Holiday weeks show elevated sales. Strong upward trend visible from 2010 to 2012.

## Chart 3 — Sales by Store Type

In [4]:
type_avg = df.groupby('Type')['Weekly_Sales'].mean().reset_index()
fig3 = px.bar(type_avg, x='Type', y='Weekly_Sales',
              title='Average Weekly Sales by Store Type',
              color='Type', color_discrete_map={'A': '#e94560', 'B': '#f39c12', 'C': '#27ae60'})
fig3.update_layout(height=450, template='plotly_white')
fig3.show()

**Key insights:** Type A stores average significantly higher sales than B and C. Store type is a strong predictor — crucial feature for our model.

## Chart 4 — Top 10 & Bottom 10 Departments

In [5]:
dept_avg = df.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False)
top10 = dept_avg.head(10)
bottom10 = dept_avg.tail(10)
combined = pd.concat([top10, bottom10]).reset_index()
combined['color'] = ['#27ae60'] * 10 + ['#e94560'] * 10
fig4 = go.Figure(go.Bar(
    x=combined['Weekly_Sales'], y=combined['Dept'].astype(str),
    orientation='h', marker_color=combined['color']
))
fig4.update_layout(title='Top 10 vs Bottom 10 Departments by Average Sales',
                   height=450, template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig4.show()

**Key insights:** Department 92 leads in average sales, while several departments have near-zero activity. Department-level filtering is essential for accurate forecasting.

## Chart 5 — Holiday Impact on Sales

In [6]:
fig5 = px.box(df, x='IsHoliday', y='Weekly_Sales',
              color='IsHoliday',
              color_discrete_map={True: '#e94560', False: '#4a9eff'},
              title='Impact of Holiday Weeks on Sales')
fig5.update_layout(height=450, template='plotly_white')
fig5.show()

**Key insights:** Holiday weeks show higher median sales and wider spread, confirming holiday effect. IsHoliday is a valuable binary feature.

## Chart 6 — Feature Correlation Heatmap

In [7]:
import plotly.figure_factory as ff
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()
fig6 = px.imshow(corr, title='Feature Correlation Heatmap',
                 color_continuous_scale='RdBu_r', aspect='auto')
fig6.update_layout(height=450, template='plotly_white')
fig6.show()

**Key insights:** Weekly_Sales correlates most with store Size and MarkDown features. MarkDown promotions show moderate positive correlation with sales.

## Chart 7 — Monthly Seasonality

In [8]:
df['month'] = df['Date'].dt.month
monthly = df.groupby('month')['Weekly_Sales'].mean().reset_index()
fig7 = px.bar(monthly, x='month', y='Weekly_Sales',
              title='Monthly Sales Seasonality',
              color='Weekly_Sales', color_continuous_scale=['#4a9eff', '#e94560'])
fig7.update_layout(height=450, template='plotly_white')
fig7.show()

**Key insights:** November and December show the highest average sales, confirming strong holiday seasonality. Summer months show moderate peaks. Week-of-year and month are critical features.

## Chart 8 — Promotional Markdowns vs Sales

In [9]:
md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
df['total_markdown'] = df[md_cols].sum(axis=1)
sampled = df.sample(min(5000, len(df)), random_state=42)
fig8 = px.scatter(sampled, x='total_markdown', y='Weekly_Sales',
                  color='IsHoliday',
                  color_discrete_map={True: '#e94560', False: '#4a9eff'},
                  title='Promotional Markdowns vs Weekly Sales',
                  opacity=0.5)
fig8.update_layout(height=450, template='plotly_white')
fig8.show()

**Key insights:** Higher markdown spend does not guarantee higher sales — the effect is noisy. However, during holiday weeks (red), markdown-boosted sales tend to be higher. Interaction terms between markdowns and holidays could be valuable.